In [ ]:
#ex_link
import pandas as pd
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from shapely.geometry import mapping
import geopandas as gpd
import cartopy.crs as ccrs
import gc
import os

In [ ]:

#add path and load data

path1=' '

in_path=path1+" "

shp = gpd.read_file("Upper_Yangtze_River.shp")

def clip_arr(data):
    data.rio.write_crs("epsg:4326", inplace=True)
    data.rio.set_spatial_dims(x_dim="longitude", y_dim="latitude", inplace=True)
    clip_data=data.rio.clip(shp.geometry, shp.crs, drop=False, all_touched=True)
    return clip_data

#clip
tp_anom=clip_arr(xr.open_dataarray("uyr_tp_anom_1979-2022.nc"))

temp_anom=clip_arr(xr.open_dataarray("uyr_2mt_anom_1979-2022.nc"))
print("clip done")

time = pd.to_datetime(temp_anom.time.values)
lon = temp_anom.longitude
lat = temp_anom.latitude
lons, lats = np.meshgrid(lon, lat)

In [ ]:
#Seasonally separated indices facilitate subsequent processing

tp_anom['time'] = pd.to_datetime(tp_anom.time.values)

def get_season_indices(data_array):
    month = data_array.time.dt.month

    seasons = {
        'Spring': [3, 4, 5],
        'Summer': [6, 7, 8],
        'Autumn': [9, 10, 11],
        'Winter': [12, 1, 2]
    }
    
    season_ind = {}
    for season, months in seasons.items():
        
        mask = month.isin(months)
        
        season_ind[season] = np.where(mask)[0]
        
        print(f"{season}: {len(season_ind[season])} days")
        
    return season_ind

# get indices
season_ind = get_season_indices(tp_anom)

for season in season_ind:
    print(f"{season} : {season_ind[season]}")

gc.collect()

In [ ]:
# total anom_quan
thre=[0.01,0.05,0.1,0.9,0.95,0.99]

def cal_anom_quantile(var, da):
    j=var[0]
    k=var[1]
    #print("j, k:", j, k)
    quan_da=da[:,j,k].quantile(thre)  
    return quan_da.values

input_combo=[]  
for j in range(len(tp_anom.latitude)):
    for k in range(len(tp_anom.longitude)):                 
            input_combo.append((j,k))

quan_tp=xr.DataArray(np.zeros((len(thre),len(tp_anom.latitude),len(tp_anom.longitude))), dims=('quantile','latitude','longitude'),
                    coords=({'longitude':tp_anom.longitude,'latitude':tp_anom.latitude,'quantile':np.array(thre)}))

quan_temp=xr.DataArray(np.zeros((len(thre),len(temp_anom.latitude),len(temp_anom.longitude))), dims=('quantile','latitude','longitude'),
                    coords=({'longitude':temp_anom.longitude,'latitude':temp_anom.latitude,'quantile':np.array(thre)}))

import tqdm

for var in tqdm.tqdm(input_combo, total=len(input_combo), position=0, leave=True):
    quan_tp[:, var[0], var[1]] = cal_anom_quantile(var, tp_anom)   
    quan_temp[:, var[0], var[1]] = cal_anom_quantile(var, temp_anom)

In [ ]:
#seasonal anom_quan
thre=[0.01,0.05,0.1,0.9,0.95,0.99]

def cal_anom_quantile(var, da):
    j=var[0]
    k=var[1]
    #print("j, k:", j, k)
    quan_prec=da[:,j,k].quantile(thre)  
    return quan_prec.values

tp_quan={}
temp_quan={}

for season, indices in season_ind.items():
    
    tp_season= tp_anom.isel(time=indices)
    temp_season= temp_anom.isel(time=indices)

    input_combo=[]  
    for j in range(len(tp_anom.latitude)):
        for k in range(len(tp_anom.longitude)):                 
                input_combo.append((j,k))

    quan_tp=xr.DataArray(np.zeros((len(thre),len(tp_anom.latitude),len(tp_anom.longitude))), dims=('quantile','latitude','longitude'),
                        coords=({'longitude':tp_anom.longitude,'latitude':tp_anom.latitude,'quantile':np.array(thre)}))

    quan_temp=xr.DataArray(np.zeros((len(thre),len(temp_anom.latitude),len(temp_anom.longitude))), dims=('quantile','latitude','longitude'),
                        coords=({'longitude':temp_anom.longitude,'latitude':temp_anom.latitude,'quantile':np.array(thre)}))

    import tqdm
    print(season,'quan cal')
    for var in tqdm.tqdm(input_combo, total=len(input_combo), position=0, leave=True):
        quan_tp[:, var[0], var[1]] = cal_anom_quantile(var, tp_season)  
        quan_temp[:, var[0], var[1]] = cal_anom_quantile(var, temp_season) 

    tp_quan[season]=quan_tp
    temp_quan[season]=quan_temp

In [ ]:
som_grids = {'grid3x4': (3, 4),}   #  'grid3x3': (3, 3), 'grid4x4': (4, 4),
grid_name='grid3x4'
grid_size=(3, 4)
thresholds=(0.5,0.75,0.9)

In [ ]:
from sklearn.linear_model import TheilSenRegressor
from pymannkendall import original_test as mk

ts_model = TheilSenRegressor(fit_intercept=True, random_state=42)
q = 0.05
vars = ['tp', 'temp']
scales = ['high', 'low']
years = np.arange(1979, 2023)

result = {}
result_yearly = {}

for grid_name, grid_size in som_grids.items():
    result[grid_name] = {}
    result_yearly[grid_name] = {}
    
    input_combo = [(j,k) for j in range(grid_size[0]) for k in range(grid_size[1])]

    classif = pd.read_csv(in_path+f"som_results/gph_{grid_name}_total_classif.csv")
    classif["time"] = time
    classif["num"] = np.arange(1, len(time)+1)
             
    for var in vars:
        print(var)
        if var == 'tp':
            quan_h, quan_l, da_anom = quan_tp.sel(quantile=(1-q)).values, quan_tp.sel(quantile=q).values, tp_anom
        else:
            quan_h, quan_l, da_anom = quan_temp.sel(quantile=(1-q)).values, quan_temp.sel(quantile=q).values, temp_anom

        for scale in scales:
            print(scale)

            occur_sum=[]
            
            for ti in range(len(time)):
                data0=da_anom[ti,:,:].values
                if scale == 'high':
                    ex_num0 = np.where(data0 - quan_h >= 0, 1, 0).astype('float64')    

                elif scale == 'low':
                    ex_num0 = np.where(data0 - quan_l <= 0, 1, 0).astype('float64')    

                occur_sum.append(clip_arr(xr.DataArray((ex_num0), dims=('latitude','longitude'), coords=({'longitude':lon,'latitude':lat}))).mean().values)

            classif['ex'] = occur_sum
                   
            group = classif.groupby('node').agg({'time': list, 'num': list, 'ex': list})
            
            result[grid_name] = pd.DataFrame(np.nan, index=range(len(input_combo)),
                                                        columns=['occur', 'slope_per', 'mk_result_per_z', 'mk_result_per_p'])
            result_yearly[grid_name] = {}

            for n in range(len(input_combo)):
                if n+1 not in group.index:
                    continue

                node_data = group.loc[n+1]
                node_df = pd.DataFrame({'time': node_data['time'], 'num': node_data['num'], 'ex': node_data['ex']})
                node_df['year'] = node_df['time'].dt.year
                
                yearly_day = node_df.groupby('year').size()
                
                node_occur = node_df['ex']
                result[grid_name].loc[n, 'occur'] = np.sum(node_occur)
                
                yearly_counts = node_df.groupby('year')['ex'].sum()
                yearly_per = yearly_counts / np.sum(node_occur)
                
                result_yearly[grid_name][n] = pd.DataFrame({
                    'yearly_counts': yearly_counts,
                    'yearly_per': yearly_per
                }).reindex(years, fill_value=0)
                
                ts_model.fit(years.reshape(-1,1), result_yearly[grid_name][n]['yearly_per'])
                slope_per = ts_model.coef_[0]
                mk_result_per = mk(result_yearly[grid_name][n]['yearly_per'].fillna(0).infer_objects(copy=False))
                
                result[grid_name].loc[n, ['slope_per', 'mk_result_per_z', 'mk_result_per_p']] = [slope_per, mk_result_per.z, mk_result_per.p]

            with pd.ExcelWriter(in_path+f"feature/{var}_ex_result_yearly_{grid_name}_total_{scale}.xlsx", engine='openpyxl') as writer:
                for key_name, df in result_yearly[grid_name].items():
                    df.to_excel(writer, sheet_name=f'node_{int(key_name)+1}', index=False)
            
            pd.DataFrame(result[grid_name]).to_csv(in_path+f"feature/{var}_ex_result_{grid_name}_total_{scale}.csv")


In [ ]:
#Calculate the number of days when the pre / temp values at each node exceed the specified percentiles.  total

from matplotlib.colors import TwoSlopeNorm

years = np.arange(1979, 2023)

q=0.05

vars=['tp','temp']

scales=['high','low']

for grid_name, grid_size in som_grids.items():
    
    node_combo=[]  
    for j in range(grid_size[0]):
        for k in range(grid_size[1]):                 
                node_combo.append((j,k))   

    
    classif=pd.read_csv(in_path+"som_results/gph_"+grid_name+"_total_classif.csv")  #pd.DataFrame
    #print(classif.shape)  (4048, 1)
    #classif.rename(columns={'x': 'node'}, inplace=True)
    classif["time"]=time.reshape((len(time),1))
    #classif['Time'] = pd.to_datetime(df['Time']) 
    classif["num"]=np.arange(1, len(time)+1, 1)

    group = classif.groupby('node').agg({'time': list, 'num': list})

    for o in range(2):
        var=vars[o]
        print(var)
        if o==0:
            quan_h=quan_tp.sel(quantile=(1-q)).values
            quan_l=quan_tp.sel(quantile=q).values
            da_anom=tp_anom
        else:
            quan_h=quan_temp.sel(quantile=(1-q)).values
            quan_l=quan_temp.sel(quantile=q).values
            da_anom=temp_anom
            
        for p in range(2):
            scale=scales[p]
            print(scale)

            occur_sum=np.zeros((len(lat),len(lon)))
            
            for ti in range(len(time)):
                data0=da_anom[ti,:,:].values
                if scale == 'high':
                    ex_num0 = np.where(data0 - quan_h >= 0, 1, 0).astype('float64')  

                elif scale == 'low':
                    ex_num0 = np.where(data0 - quan_l <= 0, 1, 0).astype('float64')  

                occur_sum += ex_num0

            sum_data=clip_arr(xr.DataArray((occur_sum), dims=('latitude','longitude'), coords=({'longitude':lon,'latitude':lat})))

            feature=pd.read_csv(in_path+f'feature/{var}_ex_result_{grid_name}_total_{scale}.csv')
            #occur_total=feature['occur'][n]

            fig, axs = plt.subplots(grid_size[0], grid_size[1], subplot_kw=dict(projection=ccrs.PlateCarree()), figsize=(12, 8))    
            for n in range(len(node_combo)):
                ex=np.zeros((len(lat),len(lon)))
                (i, j) = node_combo[n]
                time_ind=np.array(group.loc[n+1,'num'])-1

                for t in time_ind:
            
                    if scale == 'high':
                        ex_num = np.where(da_anom[t,:,:].values - quan_h >= 0, 1, 0).astype('float64')  
                        if var=='tp':
                            cmap="Blues"
                        elif var=='temp':
                            cmap="OrRd"

                    elif scale == 'low':
                        ex_num = np.where(da_anom[t,:,:].values - quan_l <= 0, 1, 0).astype('float64')  
                        if var=='tp':
                            cmap="OrRd"
                        elif var=='temp':
                            cmap="Blues"

                    ex+=ex_num
                    
                node_data=xr.DataArray((ex), dims=('latitude','longitude'), coords=({'longitude':lon,'latitude':lat}))
                node_da=clip_arr(node_data)
                node_mean=np.mean(node_da)
                levels = np.arange(0, 20+1, 1)
                ax=axs[i,j]
                ax.set_extent([90, 112, 36, 24], crs=ccrs.PlateCarree())
                cf = ax.contourf(lons, lats, node_da*100/sum_data, transform=ccrs.PlateCarree(), levels=levels, extend='max', cmap=cmap)
                #cc = ax.contour(lons, lats, node_da, transform=ccrs.PlateCarree(), levels=levels, colors='black', linewidths=0.2)
                ax.annotate(f'#{n+1}', xy=(0.98, 0.8), xycoords='axes fraction',horizontalalignment='right',fontsize=9,color='black',bbox=dict(facecolor='lightgrey', edgecolor='None', alpha=0.8, pad=0.5))
                ax.annotate(f'{node_mean:.2f}', xy=(0.01, 0.26), xycoords='axes fraction',horizontalalignment='left',fontsize=7.5,color='black',bbox=dict(facecolor='lightgrey', edgecolor='None', alpha=0.8, pad=0.5))
                ax.annotate(f'{(node_mean*100/np.mean(sum_data)):.2f}%', xy=(0.01, 0.055), xycoords='axes fraction',horizontalalignment='left',fontsize=7.5,color='black',bbox=dict(facecolor='lightgrey', edgecolor='None', alpha=0.8, pad=0.5))
                ax.add_geometries(shp.geometry, crs=ccrs.PlateCarree(), facecolor='none', edgecolor='black', alpha=1, linewidth=0.2)

                # 设置子图边框宽度
                for spine in ax.spines.values():
                    spine.set_linewidth(0.2)

                mk_result_per_p=feature['mk_result_per_p'][n]
                slope_per=feature['slope_per'][n]

                if slope_per<0:
                    color='blue'
                elif slope_per>0:
                    color='red'
                
                if mk_result_per_p<0.01:
                    text='**'
                elif mk_result_per_p<0.05:
                    text='*'
                else:
                    text=''
                    
                ax.annotate(f'{slope_per*1000:.4f}', xy=(0.98, 0.04), xycoords='axes fraction',horizontalalignment='right',fontsize=6,color=color)
                ax.annotate(text, xy=(0.98, 0.14), xycoords='axes fraction',horizontalalignment='right',fontsize=9,color=color)

            fig.subplots_adjust(wspace=0.05, hspace=0.05) 
            plt.subplots_adjust(left=0.3, right=0.7, top=0.7, bottom=0.38) 

            cbar = fig.colorbar(cf, ax=axs[:,:], orientation='horizontal', aspect=60, pad=0.03)
            cbar.set_ticks(np.arange(0, 20+5, 5))
            cbar.outline.set_linewidth(0.2)
            cbar.ax.tick_params(labelsize=6.5, length=2,width=0.2)
            fig.text(0.69,0.399,'%',size=6.5)

            plt.show()
            fig.savefig(in_path+"extreme_fig/"+grid_name+f"_{var}_anom_extremes_{scale}_0.05_total.png",dpi=1000,bbox_inches='tight')

In [ ]:
#feature  seasonal
from sklearn.linear_model import TheilSenRegressor
from pymannkendall import original_test as mk
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
                        
ts_model = TheilSenRegressor(fit_intercept=True, random_state=42)
q = 0.05
vars = ['tp', 'temp']
scales = ['high', 'low']
years = np.arange(1979, 2023)

result = {}
result_yearly = {}

for grid_name, grid_size in som_grids.items():
    result[grid_name] = {}
    result_yearly[grid_name] = {}
    
    input_combo = [(j,k) for j in range(grid_size[0]) for k in range(grid_size[1])]

    classif = pd.read_csv(in_path+f"som_results/gph_{grid_name}_total_classif.csv")
    classif["time"] = time
    classif["num"] = np.arange(1, len(time)+1)
                
    for season, indices in season_ind.items():
        print(season)
        season_class = classif.iloc[indices].copy()

        for var in vars:
            print(var)
            if var == 'tp':
                quan_h, quan_l, da_anom = tp_quan[season].sel(quantile=(1-q)).values, tp_quan[season].sel(quantile=q).values, tp_anom
            else:
                quan_h, quan_l, da_anom = temp_quan[season].sel(quantile=(1-q)).values, temp_quan[season].sel(quantile=q).values, temp_anom

            for scale in scales:
                print(scale)

                occur_sum=[]
                
                for t in range(len(indices)):
        
                    ti=indices[t]
                
                    data0=da_anom[ti,:,:].values
                    
                    if scale == 'high':
                        ex_num0 = np.where(data0 - quan_h >= 0, 1, 0).astype('float64')  

                    elif scale == 'low':
                        ex_num0 = np.where(data0 - quan_l <= 0, 1, 0).astype('float64')    

                    occur_sum.append(clip_arr(xr.DataArray((ex_num0), dims=('latitude','longitude'), coords=({'longitude':lon,'latitude':lat}))).mean().values)

                season_class['ex'] = np.array(occur_sum)

                group = season_class.groupby('node').agg({'time': list, 'num': list, 'ex': list})
                
                result[grid_name][season] = pd.DataFrame(np.nan, index=range(len(input_combo)),
                                                            columns=['occur', 'slope_per', 'mk_result_per_z', 'mk_result_per_p'])
                result_yearly[grid_name][season] = {}

                for n in range(len(input_combo)):
                    if n+1 not in group.index:
                        continue

                    node_data = group.loc[n+1]
                    node_df = pd.DataFrame({'time': node_data['time'], 'num': node_data['num'], 'ex': node_data['ex']})
                    node_df['year'] = node_df['time'].dt.year
                    
                    yearly_day = node_df.groupby('year').size()
                                
                    node_occur = node_df['ex']
                    result[grid_name][season].loc[n, 'occur'] = np.sum(node_occur)

                    yearly_counts = node_df.groupby('year')['ex'].sum()
                    yearly_per = yearly_counts / np.sum(node_occur)
                    
                    result_yearly[grid_name][season][n] = pd.DataFrame({
                        'yearly_counts': yearly_counts,
                        'yearly_per': yearly_per
                    }).reindex(years, fill_value=0)
                    
                    # Theil-Sen
                    ts_model.fit(years.reshape(-1,1), result_yearly[grid_name][season][n]['yearly_per'])
                    slope_per = ts_model.coef_[0]
                    mk_result_per = mk(result_yearly[grid_name][season][n]['yearly_per'].fillna(0))
                    
                    result[grid_name][season].loc[n, ['slope_per', 'mk_result_per_z', 'mk_result_per_p']] = [slope_per, mk_result_per.z, mk_result_per.p]

                # save 
                with pd.ExcelWriter(in_path+f"feature/{var}_ex_result_yearly_{grid_name}_{season}_{scale}.xlsx", engine='openpyxl') as writer:
                    for key_name, df in result_yearly[grid_name][season].items():
                        df.to_excel(writer, sheet_name=f'node_{int(key_name)+1}', index=False)
                
                pd.DataFrame(result[grid_name][season]).to_csv(in_path+f"feature/{var}_ex_result_{grid_name}_{season}_{scale}.csv")


In [ ]:
#Calculate the number of days when the pre / temp values at each node exceed the specified percentiles.   season

from matplotlib.colors import TwoSlopeNorm

years = np.arange(1979, 2023)

q=0.05

vars=['tp','temp']

scales=['high','low']

def clip_arr(arr):
    data=arr.rename({'latitude': 'lat', 'longitude': 'lon'})
    data.rio.write_crs("epsg:4326", inplace=True)
    data.rio.set_spatial_dims(x_dim="lon", y_dim="lat", inplace=True)
    clip_data=data.rio.clip(shp.geometry, shp.crs, drop=False, all_touched=True)
    return clip_data

for grid_name, grid_size in som_grids.items():
    
    node_combo=[]  
    for j in range(grid_size[0]):
        for k in range(grid_size[1]):                 
                node_combo.append((j,k))   

    season_classif = {}

    classif=pd.read_csv(in_path+"som_results/gph_"+grid_name+"_total_classif.csv")  #pd.DataFrame
    #print(classif.shape)  (4048, 1)
    #classif.rename(columns={'x': 'node'}, inplace=True)
    classif["time"]=time.reshape((len(time),1))
    #classif['Time'] = pd.to_datetime(df['Time'])  
    classif["num"]=np.arange(1, len(time)+1, 1)

    for season, indices in season_ind.items():

        len_season=len(indices)
        
        season_class = pd.DataFrame(columns=classif.columns)
        
        for index, row in classif.iterrows():
            node_num = np.array(row['num'])-1
            mask = np.isin(node_num, indices)
            if np.any(mask):
                new_row = {}
                for col in classif.columns:
                    if isinstance(row[col], list):
                        new_row[col] = np.array(row[col])[mask].tolist()
                    else:
                        new_row[col] = row[col]
                
                season_class.loc[index] = new_row
        
        season_classif[season] = season_class
        print(season)

        group = season_classif[season].groupby('node').agg({'time': list, 'num': list})
       
        for o in range(2):
            var=vars[o]
            print(var)
            if o==0:
                quan_h=tp_quan[season].sel(quantile=(1-q)).values
                quan_l=tp_quan[season].sel(quantile=q).values
                da_anom=tp_anom
                if season=='Summer':
                    levels=np.arange(0,15+0.75,0.75)
                    clevel=np.arange(0,15+1,5)
                else:
                    levels=np.arange(0,6+0.3,0.3)
                    clevel=np.arange(0,6+1,2)
            else:
                quan_h=temp_quan[season].sel(quantile=(1-q)).values
                quan_l=temp_quan[season].sel(quantile=q).values
                da_anom=temp_anom
                if season=='Summer' or season=='Autumn':
                    levels=np.arange(0,6+0.3,0.3)
                    clevel=np.arange(0,6+1,2)
                else:
                    levels=np.arange(0,15+0.75,0.75)
                    clevel=np.arange(0,15+1,5)
                
            for p in range(2):
                scale=scales[p]
                print(scale)

                occur_sum=np.zeros((len(lat),len(lon)))
                
                for ti in range(len(time)):
                    data0=da_anom[ti,:,:].values
                    if scale == 'high':
                        ex_num0 = np.where(data0 - quan_h >= 0, 1, 0).astype('float64')  

                    elif scale == 'low':
                        ex_num0 = np.where(data0 - quan_l <= 0, 1, 0).astype('float64')  

                    occur_sum += ex_num0

                sum_data=clip_arr(xr.DataArray((occur_sum), dims=('latitude','longitude'), coords=({'longitude':lon,'latitude':lat})))


                feature_season=pd.read_csv(in_path+f'feature/{var}_ex_result_{grid_name}_{season}_{scale}.csv')
                
                fig, axs = plt.subplots(grid_size[0], grid_size[1], subplot_kw=dict(projection=ccrs.PlateCarree()), figsize=(12, 8))    
                for n in range(len(node_combo)):
                    ex=np.zeros((len(lat),len(lon)))
                    (i, j) = node_combo[n]
                    time_ind=np.array(group.loc[n+1,'num'])-1

                    for t in time_ind:
                        da_anom_=da_anom[t,:,:].values
                        if scale == 'high':
                            ex_num = np.where(da_anom_ - quan_h >= 0, 1, 0).astype('float64')  
                            if var=='tp':
                                cmap="Blues"
                            elif var=='temp':
                                cmap="OrRd"

                        elif scale == 'low':
                            ex_num = np.where(da_anom_ - quan_l <= 0, 1, 0).astype('float64')  
                            if var=='tp':
                                cmap="OrRd"
                            elif var=='temp':
                                cmap="Blues"

                        ex+=ex_num
                        
                    node_data=xr.DataArray((ex), dims=('latitude','longitude'), coords=({'longitude':lon,'latitude':lat}))
                    node_da=clip_arr(node_data)
                    node_mean=np.mean(node_da)
                    #levels = np.arange(0, 12+0.6, 0.6)
                    ax=axs[i,j]
                    ax.set_extent([90, 112, 36, 24], crs=ccrs.PlateCarree())
                    cf = ax.contourf(lons, lats, node_da*100/sum_data, transform=ccrs.PlateCarree(), levels=levels, extend='max', cmap=cmap)
                    #cc = ax.contour(lons, lats, node_da, transform=ccrs.PlateCarree(), levels=levels, colors='black', linewidths=0.2)
                    ax.annotate(f'#{n+1}', xy=(0.98, 0.8), xycoords='axes fraction',horizontalalignment='right',fontsize=9,color='black',bbox=dict(facecolor='lightgrey', edgecolor='None', alpha=0.8, pad=0.5))
                    ax.annotate(f'{node_mean:.2f}', xy=(0.01, 0.26), xycoords='axes fraction',horizontalalignment='left',fontsize=7.5,color='black',bbox=dict(facecolor='lightgrey', edgecolor='None', alpha=0.8, pad=0.5))
                    ax.annotate(f'{(node_mean*100/np.mean(sum_data)):.2f}%', xy=(0.01, 0.055), xycoords='axes fraction',horizontalalignment='left',fontsize=7.5,color='black',bbox=dict(facecolor='lightgrey', edgecolor='None', alpha=0.8, pad=0.5))
                    ax.add_geometries(shp.geometry, crs=ccrs.PlateCarree(), facecolor='none', edgecolor='black', alpha=1, linewidth=0.2)

                    for spine in ax.spines.values():
                        spine.set_linewidth(0.2)

                    mk_result_per_p=feature_season['mk_result_per_p'][n]
                    slope_per=feature_season['slope_per'][n]

                    if slope_per<0:
                        color='blue'
                    elif slope_per>0:
                        color='red'
                    
                    if mk_result_per_p<0.01:
                        text='**'
                    elif mk_result_per_p<0.05:
                        text='*'
                    else:
                        text=''
                        
                    ax.annotate(f'{slope_per*1000:.4f}', xy=(0.98, 0.04), xycoords='axes fraction',horizontalalignment='right',fontsize=6,color=color)
                    ax.annotate(text, xy=(0.98, 0.14), xycoords='axes fraction',horizontalalignment='right',fontsize=9,color=color)

                fig.subplots_adjust(wspace=0.05, hspace=0.05) 
                plt.subplots_adjust(left=0.3, right=0.7, top=0.7, bottom=0.38) 

                cbar = fig.colorbar(cf, ax=axs[:,:], orientation='horizontal', aspect=60, pad=0.03)
                cbar.set_ticks(clevel)
                cbar.outline.set_linewidth(0.2)
                cbar.ax.tick_params(labelsize=6.5, length=2,width=0.2)
                fig.text(0.69,0.399,'%',size=6.5)

                plt.show()
                fig.savefig(in_path+"extreme_fig/"+grid_name+f"_{var}_anom_extremes_{scale}_0.05_{season}.png",dpi=1000,bbox_inches='tight')